# Aula 12 - Notebook: Matriz de Incidência e Balanço de Massa Matricial

Neste notebook implementamos a **Matriz de Incidência Vértice-Aresta** $B \in \{-1, 0, 1\}^{n 	imes m}$ e resolvemos o balanço de massa matricial $B \cdot ec{Q} = ec{S}$ para a rede de fluidos.


In [1]:
def formatar_tabela(dados):
    """Formata lista de dicionarios em tabela ASCII pura."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)

def formatar_matriz(matriz, rotulos_linhas, rotulos_cols):
    """Formata matriz 2D em tabela ASCII pura."""
    larguras = [max(len(str(r)), 6) for r in rotulos_cols]
    larg_linha = max(len(str(r)) for r in rotulos_linhas)
    header = f"{' ' * larg_linha} | " + " | ".join(f"{c:>{larguras[j]}}" for j, c in enumerate(rotulos_cols))
    divisor = f"{'-' * larg_linha}-+-" + "-+-".join("-" * larguras[j] for j in range(len(rotulos_cols)))
    linhas = [header, divisor]
    for i, r_nome in enumerate(rotulos_linhas):
        vals = []
        for j in range(len(rotulos_cols)):
            v = matriz[i][j]
            v_str = "∞" if v == float('inf') else str(v)
            vals.append(f"{v_str:>{larguras[j]}}")
        linhas.append(f"{r_nome:<{larg_linha}} | " + " | ".join(vals))
    return "\n".join(linhas)

class GrafoTubulacao:
    def __init__(self, vertices):
        self.vertices = vertices
        self.v_to_idx = {v: i for i, v in enumerate(vertices)}
        self.idx_to_v = {i: v for i, v in enumerate(vertices)}
        self.n = len(vertices)
        self.adj_binaria = [[0] * self.n for _ in range(self.n)]
        self.adj_pesos = [[float('inf')] * self.n for _ in range(self.n)]
        for i in range(self.n): self.adj_pesos[i][i] = 0.0
        self.arestas_detalhes = []

    def adicionar_tubulacao(self, origem, destino, comprimento_m, tag_valvula, diametro_pol=4.0):
        u = self.v_to_idx[origem]
        v = self.v_to_idx[destino]
        self.adj_binaria[u][v] = 1
        self.adj_pesos[u][v] = comprimento_m
        self.arestas_detalhes.append({
            "Origem": origem, "Destino": destino,
            "Comprimento (m)": comprimento_m, "Válvula ISA": tag_valvula, "Diâmetro (pol)": diametro_pol
        })

def criar_rede_padrao():
    nos = ["TK-301_NH3", "TK-302_H3PO4", "MAN-101", "P-101", "P-102", "R-101", "TK-303_Pulmao", "GRAN-201"]
    g = GrafoTubulacao(nos)
    g.adicionar_tubulacao("TK-301_NH3", "MAN-101", 15.0, "XV-301", 3.0)
    g.adicionar_tubulacao("TK-302_H3PO4", "MAN-101", 12.0, "XV-302", 4.0)
    g.adicionar_tubulacao("MAN-101", "P-101", 8.0, "XV-101A", 4.0)
    g.adicionar_tubulacao("MAN-101", "P-102", 10.0, "XV-101B", 4.0)
    g.adicionar_tubulacao("P-101", "R-101", 25.0, "XV-102A", 4.0)
    g.adicionar_tubulacao("P-102", "R-101", 22.0, "XV-102B", 4.0)
    g.adicionar_tubulacao("R-101", "GRAN-201", 30.0, "XV-201", 6.0)
    g.adicionar_tubulacao("R-101", "TK-303_Pulmao", 18.0, "XV-202", 6.0)
    g.adicionar_tubulacao("TK-303_Pulmao", "GRAN-201", 20.0, "XV-203", 6.0)
    return g

rede = criar_rede_padrao()

class CalculadorIncidencia:
    @staticmethod
    def construir_matriz_incidencia(vertices, arestas):
        n = len(vertices)
        m = len(arestas)
        v_idx = {v: i for i, v in enumerate(vertices)}
        B = [[0] * m for _ in range(n)]
        nomes_e = []
        for j, a in enumerate(arestas):
            u = v_idx[a["Origem"]]
            v = v_idx[a["Destino"]]
            B[u][j] = -1
            B[v][j] = 1
            nomes_e.append(f"e{j+1}:{a['Origem']}->{a['Destino']}")
        return B, nomes_e

B_mat, nomes_arestas = CalculadorIncidencia.construir_matriz_incidencia(rede.vertices, rede.arestas_detalhes)
print("=== MATRIZ DE INCIDÊNCIA B ===")
print(formatar_matriz(B_mat, rede.vertices, [f"e{j+1}" for j in range(len(rede.arestas_detalhes))]))

# Verificação soma por coluna nula
somas_col = [sum(B_mat[i][j] for i in range(len(rede.vertices))) for j in range(len(rede.arestas_detalhes))]
assert all(s == 0 for s in somas_col)

# Balanço de massa Q (m3/h)
Q_vec = [50.0, 70.0, 120.0, 0.0, 120.0, 0.0, 120.0, 0.0, 0.0]
S_res = []
for i, v_nome in enumerate(rede.vertices):
    balanco = sum(B_mat[i][j] * Q_vec[j] for j in range(len(Q_vec)))
    S_res.append({"Nó": v_nome, "Balanço Líquido (m3/h)": f"{balanco:.1f}"})

print("\n--- Balanço de Massa em Regime Permanente (S = B * Q) ---")
print(formatar_tabela(S_res))


=== MATRIZ DE INCIDÊNCIA B ===
              |     e1 |     e2 |     e3 |     e4 |     e5 |     e6 |     e7 |     e8 |     e9
--------------+--------+--------+--------+--------+--------+--------+--------+--------+-------
TK-301_NH3    |     -1 |      0 |      0 |      0 |      0 |      0 |      0 |      0 |      0
TK-302_H3PO4  |      0 |     -1 |      0 |      0 |      0 |      0 |      0 |      0 |      0
MAN-101       |      1 |      1 |     -1 |     -1 |      0 |      0 |      0 |      0 |      0
P-101         |      0 |      0 |      1 |      0 |     -1 |      0 |      0 |      0 |      0
P-102         |      0 |      0 |      0 |      1 |      0 |     -1 |      0 |      0 |      0
R-101         |      0 |      0 |      0 |      0 |      1 |      1 |     -1 |     -1 |      0
TK-303_Pulmao |      0 |      0 |      0 |      0 |      0 |      0 |      0 |      1 |     -1
GRAN-201      |      0 |      0 |      0 |      0 |      0 |      0 |      1 |      0 |      1

--- Balanço de Mas